In [ ]:
import zipfile
import os

zip_path = "/content/gemma-2-sentiment-adapter.zip"
extract_path = "/content/gemma-2-sentiment-adapter"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction completed!")

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
model_name = 'google/gemma-2b-it'
num_classes = 3
max_seq_length = 512

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
train_path = "/content/twitter_training_3class.csv"
valid_path = "/content/twitter_valid_3class.csv"
test_path = "/content/twitter_testing_3class.csv"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "right"

In [ ]:
system_prompt = """Classify sentiment towards the 'Target Entity' as [Others, Negative, Positive].

Focus exclusively on emotion directed AT the Target Entity:
- Positive: The text expresses explicit praise, enjoyment, or uses positive gaming slang ("killed it") directed specifically at the Target Entity.
- Negative: The text expresses direct complaints, deep frustration, or insults directed specifically at the Target Entity.

If it is not clearly Positive or Negative toward the Target Entity, classify it as:
- Others: The catch-all category. Use this if the text is objective/neutral, if the Target Entity is missing entirely, OR if the text contains strong emotions (like "love", "hate", or "thank you") that are directed at someone or something else."""

In [ ]:
def format_with_system_prompt(row):
    user_text = row['tweet']

    # Extract the entity from your dataset
    # (Make sure 'entity' matches the exact column name in your CSV)
    target_entity = row['entity']

    # Prepend the system prompt, the specific entity, and the text
    full_text = f"{system_prompt}\n\nTarget Entity: {target_entity}\nText: {user_text}"

    return {"formatted_text": full_text}

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset(
    "csv",
    data_files={
        "train": train_path,
        "validation": valid_path,
        "test": test_path
    }
)

print(raw_datasets)

In [ ]:
formatted_datasets = raw_datasets.map(format_with_system_prompt)

In [ ]:
def tokenize_function(examples):

    tokenized_inputs = tokenizer(
        examples["formatted_text"],
        truncation=True,
        max_length=308
    )

    tokenized_inputs["labels"] = examples["sentiment"]

    return tokenized_inputs

In [ ]:
tokenized_datasets = formatted_datasets.map(tokenize_function, batched=True)

cols_to_remove = formatted_datasets["train"].column_names
tokenized_datasets = tokenized_datasets.remove_columns(cols_to_remove)

print(tokenized_datasets["train"].column_names)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_classes,
    quantization_config=bnb_config,
    device_map={"": 0}
)

In [ ]:
model.config.pad_token_id = tokenizer.pad_token_id
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj","gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.2,
    bias="none",
    task_type="SEQ_CLS"
)

In [ ]:
model = get_peft_model(model, lora_config)
model.config.use_cache = False

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, predictions)}

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/gemma_sentiment_model",

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    num_train_epochs=6,
    weight_decay=0.01,
    max_grad_norm=0.3,

    # Evaluation and Checkpointing
    logging_strategy="steps",
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
trainer.train()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

In [ ]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    model,
    "/content/gemma-2-sentiment-adapter"
)

In [ ]:
eval_dataloader = DataLoader(
    tokenized_datasets["test"],
    batch_size=8,
    collate_fn=data_collator
)

In [ ]:
all_predictions = []
all_true_labels = []
all_probs = []

model.eval()

with torch.no_grad():
    for batch in tqdm(eval_dataloader, desc="Evaluating"):

        input_ids = batch['input_ids'].to(model.device)
        attention_mask = batch['attention_mask'].to(model.device)
        labels = batch['labels'].to(model.device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(logits, dim=-1)

        all_predictions.extend(preds.cpu().numpy())
        all_true_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

In [ ]:
print("Predictions:", len(all_predictions))
print("True labels:", len(all_true_labels))
print("Probabilities:", len(all_probs))

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
import numpy as np

y_true = np.array(all_true_labels)
y_pred = np.array(all_predictions)

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')
roc_auc = roc_auc_score(y_true, np.array(all_probs), multi_class='ovr', average='weighted')


print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")

In [ ]:
eval_df = pd.read_csv('/content/twitter_testing_3class.csv')

In [ ]:
label_map = {0: "Other", 1: "Negative", 2: "Positive"}

results_df = pd.DataFrame({
    'tweet_text': eval_df['tweet'].tolist(),
    'true_label_num': all_true_labels,
    'predicted_label_num': all_predictions
})

errors_df = results_df[results_df['true_label_num'] != results_df['predicted_label_num']]

print(f"Total Evaluation Size: {len(results_df)}")
print(f"Total Errors Made: {len(errors_df)}")
print(f"Accuracy: {((len(results_df) - len(errors_df)) / len(results_df)) * 100:.2f}%\n")
print("="*50)
print("EXAMPLES OF INCORRECT PREDICTIONS ")
print("="*50)

for index, row in errors_df.head(20).iterrows():
    actual_str = label_map.get(row['true_label_num'], "Unknown")
    pred_str = label_map.get(row['predicted_label_num'], "Unknown")

    print(f"TEXT:     {row['tweet_text']}")
    print(f"ACTUAL:   {actual_str}")
    print(f"PREDICTED: {pred_str}")
    print("-" * 50)

In [ ]:
import torch.nn.functional as F

print("Running evaluation to capture all probabilities...")
model.eval()

all_predictions = []
all_true_labels = []
all_probabilities = []

with torch.no_grad():
    for batch in tqdm(eval_dataloader, desc="Evaluating"):
        input_ids = batch['input_ids'].to(model.device)
        attention_mask = batch['attention_mask'].to(model.device)
        labels = batch['labels'].to(model.device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        probs = F.softmax(logits, dim=-1)

        predictions = torch.argmax(probs, dim=-1)

        all_predictions.extend(predictions.cpu().numpy())
        all_true_labels.extend(labels.cpu().numpy())
        all_probabilities.extend(probs.cpu().numpy())

label_map = {0: "Other", 1: "Negative", 2: "Positive"}


results_df = pd.DataFrame({
    'tweet_text': eval_df['tweet'].tolist(),
    'actual': [label_map.get(label, "Unknown") for label in all_true_labels],
    'predicted': [label_map.get(pred, "Unknown") for pred in all_predictions],
    'probs': all_probabilities # This holds [score_0, score_1, score_2, score_3]
})

errors_df = results_df[results_df['actual'] != results_df['predicted']]

print("\n" + "="*80)
print(" MISCLASSIFIED EXAMPLES WITH FULL CONFIDENCE SPREAD ")
print("="*80)

for index, row in errors_df.head(20).iterrows():
    p = row['probs']

    print(f"TEXT:       {row['tweet_text']}")
    print(f"ACTUAL:     {row['actual']}")
    print(f"PREDICTED:  {row['predicted']}")

    print("SCORES:     "
          f"Other: {p[0]:>6.2%} | "
          f"Negative: {p[1]:>6.2%} | "
          f"Positive: {p[2]:>6.2%} | ")
    print("-" * 80)

In [ ]:
pos_to_neg_errors = results_df[(results_df['actual'] == 'Positive') & (results_df['predicted'] == 'Negative')]

neg_to_pos_errors = results_df[(results_df['actual'] == 'Negative') & (results_df['predicted'] == 'Positive')]


def print_error_examples(df, title, num_examples=20):
    print("\n" + "="*80)
    print(f" {title} (Total found: {len(df)})")
    print("="*80)

    for index, row in df.head(num_examples).iterrows():
        p = row['probs']

        print(f"TEXT:       {row['tweet_text']}")
        print(f"ACTUAL:     {row['actual']}")
        print(f"PREDICTED:  {row['predicted']}")
        print("SCORES:     "
              f"Other: {p[0]:>6.2%} | "
              f"Negative: {p[1]:>6.2%} | "
              f"Positive: {p[2]:>6.2%} | ")
        print("-" * 80)

print_error_examples(pos_to_neg_errors, "ACTUAL: POSITIVE -> PREDICTED: NEGATIVE", 20)

print_error_examples(neg_to_pos_errors, "ACTUAL: NEGATIVE -> PREDICTED: POSITIVE", 20)

In [ ]:
other_to_neg_errors = results_df[(results_df['actual'] == 'Other') & (results_df['predicted'] == 'Negative')]

other_to_pos_errors = results_df[(results_df['actual'] == 'Other') & (results_df['predicted'] == 'Positive')]


def print_error_examples(df, title, num_examples=20):
    print("\n" + "="*80)
    print(f" {title} (Total found: {len(df)})")
    print("="*80)

    for index, row in df.head(num_examples).iterrows():
        p = row['probs']

        print(f"TEXT:       {row['tweet_text']}")
        print(f"ACTUAL:     {row['actual']}")
        print(f"PREDICTED:  {row['predicted']}")
        print("SCORES:     "
              f"Other: {p[0]:>6.2%} | "
              f"Negative: {p[1]:>6.2%} | "
              f"Positive: {p[2]:>6.2%} | ")
        print("-" * 80)

print_error_examples(other_to_neg_errors, "ACTUAL: Other -> PREDICTED: NEGATIVE", 20)

print_error_examples(other_to_pos_errors, "ACTUAL: Other -> PREDICTED: POSITIVE", 20)

In [ ]:
class_names = ["Other","Negative", "Positive"]

print("--- Classification Report ---")
report = classification_report(
    all_true_labels,
    all_predictions,
    target_names=class_names,
    digits=3
)
print(report)

In [ ]:
cm = confusion_matrix(all_true_labels, all_predictions)
plt.figure(figsize=(8, 6))

sns.heatmap(cm, annot=True,fmt='d',   xticklabels=class_names, yticklabels=class_names)

plt.title('Validation Confusion Matrix', fontsize=16, pad=20)
plt.xlabel('Predicted Sentiment', fontsize=12, labelpad=10)
plt.ylabel('True Sentiment', fontsize=12, labelpad=10)

plt.tight_layout()
plt.show()

In [ ]:
adapter_path = "/content/gemma-2-sentiment-adapter"

# Save only the LoRA weights and configuration
model.save_pretrained(adapter_path)

In [ ]:
import shutil

folder_path = "/content/gemma-2-sentiment-adapter"
zip_path = "/content/gemma-2-sentiment-adapter"

shutil.make_archive(zip_path.replace('.zip', ''), 'zip', folder_path)

print("Zipped successfully!")